# Data Wrangling: Acne Detection System & Skincare Recommendation
**Tim ID:** CC26-PSU132

Notebook ini berisi proses pengolahan data untuk sistem rekomendasi skincare. Proses yang dilakukan meliputi:
1. **Gathering Data**: Mengambil data dari repositori GitHub.
2. **Assessing Data**: Menilai kualitas data (cek missing values, duplikat, dll).
3. **Cleaning Data**: Membersihkan dan menstandarisasi data.
4. **Feature Engineering**: Membuat logika rekomendasi berdasarkan tingkat keparahan jerawat.
5. **Export**: Menyimpan data matang untuk digunakan oleh tim Full-stack dan Dashboard.

In [ ]:
# Mengimpor library yang dibutuhkan untuk pengolahan data tabular
import pandas as pd
import numpy as np

# 🌟 KODE BARU: Import library untuk visualisasi data (EDA)
import matplotlib.pyplot as plt
import seaborn as sns

# Setup tema visualisasi agar rapi
sns.set_theme(style="whitegrid")

# Mengatur tampilan agar kolom tidak terpotong saat ditampilkan
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

print("Semua Library berhasil diimpor!")

Semua Library berhasil diimpor!


## 1. Gathering Data
Pada tahap ini, kita akan membaca data mentah (*raw data*) yang disimpan di GitHub.
* **dataset_skic.csv**: Data produk skincare (Brand, Nama Produk, Bahan Aktif).
* **ingredients_category.csv**: Data referensi kandungan (Fungsi bahan, tingkat keamanan/warning).

In [ ]:
# Tautan raw GitHub yang telah dipersiapkan
url_produk = 'https://raw.githubusercontent.com/lalamghf/capstone-project/refs/heads/main/dataset_rekomendasi/dataset_skic.csv'
url_ingredients = 'https://raw.githubusercontent.com/lalamghf/capstone-project/refs/heads/main/dataset_rekomendasi/ingredients_category.csv'

# Membaca data ke dalam DataFrame Pandas
df_produk = pd.read_csv(url_produk)
df_ingredients = pd.read_csv(url_ingredients)

# Menggunakan display() untuk menampilkan kedua tabel sekaligus
print("--- Cuplikan Data Produk ---")
display(df_produk.head(5))

print("\n--- Cuplikan Data Ingredients ---")
display(df_ingredients.head(5))

--- Cuplikan Data Produk ---


,ID,Brand,Produk,Jenis Produk,Untuk Kulit,Masalah Kulit,Ukuran,Tipe Bahan Aktif,Tahun Rilis
0,1,Avoskin,Tea Tree Spot Gel,Essence,Berminyak,Jerawat,100 ml,Bakuchiol,2021
1,2,Dear Me Beauty,Bakuchiol Night Cream,Essence,Kering,Dehidrasi,50 ml,Hyaluronic Acid,2023
2,3,Somethinc,Glow Boost Serum,Gel,Berminyak,Flek hitam,30 ml,Salicylic Acid,2018
3,4,COSRX,Acne Treatment Serum,Cleanser,Berminyak,Iritasi,100 ml,Bakuchiol,2018
4,5,The Ordinary,Tea Tree Spot Gel,Sunscreen,Berminyak,Kusam,100 ml,Hyaluronic Acid,2022



--- Cuplikan Data Ingredients ---


,ingredient_name,function1,function2,warning1,warning2,ingredient_origin,ingredient_charge
0,"1,2-Hexanediol",Solvent,Preservative Booster,NaN,NaN,Synthetic,Non-ionik
1,"2,3-Butanediol",Humectant,Solvent,NaN,NaN,Synthetic,Non-ionik
2,2-Aminobutanol,pH Adjuster,NaN,Irritant,NaN,Synthetic,Kationik
3,2-Methylpropanediol,Solvent,Humectant,NaN,NaN,Synthetic,Non-ionik
4,2-O-Ethyl Ascorbic Acid,Antioxidant,Brightening,NaN,NaN,Synthetic,Non-ionik


In [ ]:
print(f"Data Produk: {df_produk.shape[0]} baris, {df_produk.shape[1]} kolom")
print(f"Data Ingredients: {df_ingredients.shape[0]} baris, {df_ingredients.shape[1]} kolom")

Data Produk: 185 baris, 9 kolom
Data Ingredients: 747 baris, 7 kolom


## 2. Assessing Data
Tahap ini bertujuan untuk memetakan kondisi dataset. Kita memeriksa tipe data, menghitung jumlah *missing values* (data kosong), dan memastikan ada tidaknya baris data yang terduplikasi.

### 2.1 Pemeriksaan Struktur dan Tipe Data
Mengevaluasi jumlah baris, kolom, dan tipe data (*integer*, *object*/*string*) pada kedua tabel untuk memastikan tidak ada kesalahan format.

In [ ]:
print("--- Ringkasan Struktur Data Produk ---")
df_produk.info()

print("\n" + "-"*30 + "\n")

print("--- Ringkasan Struktur Data Ingredients ---")
df_ingredients.info()

--- Ringkasan Struktur Data Produk ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 185 entries, 0 to 184
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   ID                185 non-null    int64 
 1   Brand             185 non-null    object
 2   Produk            185 non-null    object
 3   Jenis Produk      185 non-null    object
 4   Untuk Kulit       185 non-null    object
 5   Masalah Kulit     185 non-null    object
 6   Ukuran            185 non-null    object
 7   Tipe Bahan Aktif  185 non-null    object
 8   Tahun Rilis       185 non-null    int64 
dtypes: int64(2), object(7)
memory usage: 13.1+ KB

------------------------------

--- Ringkasan Struktur Data Ingredients ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 747 entries, 0 to 746
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   ingredient_name    7

### 📌 Insight Pemeriksaan Struktur
Secara umum, tipe data (Dtypes) tidak bermasalah. Namun, keberadaan kolom `ID` dan `Tahun Rilis` pada dataset produk dianggap tidak relevan untuk logika klasifikasi keparahan jerawat, sehingga kedua kolom ini menjadi target penghapusan pada tahap *Cleaning*.

### 2.2 Pemeriksaan Missing Values (Nilai Kosong)
Mendeteksi kolom mana saja yang memiliki nilai *Null* atau *NaN* (Not a Number). Keberadaan data kosong dapat membuat sistem rekomendasi *error* (*crash*) saat dijalankan oleh tim Back-End.

In [ ]:
# Mengecek jumlah nilai yang hilang di setiap kolom
print("Jumlah Missing Values pada Data Produk:")
print(df_produk.isna().sum())

print("\nJumlah Missing Values pada Data Ingredients:")
print(df_ingredients.isna().sum())

Jumlah Missing Values pada Data Produk:
ID                  0
Brand               0
Produk              0
Jenis Produk        0
Untuk Kulit         0
Masalah Kulit       0
Ukuran              0
Tipe Bahan Aktif    0
Tahun Rilis         0
dtype: int64

Jumlah Missing Values pada Data Ingredients:
ingredient_name        0
function1              0
function2            549
warning1             605
warning2             698
ingredient_origin      0
ingredient_charge      0
dtype: int64


### 📌 Insight Missing Values
Data Produk memiliki kelengkapan 100%. Kekosongan pada Data Ingredients (seperti kolom `warning1` dan `function2`) adalah hal yang **logis**, karena tidak semua bahan aktif memiliki efek samping peringatan atau fungsi ganda. Nilai kosong ini akan diisi (imputasi) di tahap *Cleaning*.

### 2.3 Pemeriksaan Duplikasi pada dataset_skic.csv dan ingredients_category.csv:
Mengecek apakah ada baris data yang dimasukkan dua kali (duplikat). Selain itu, kita melihat deskripsi statistik untuk memahami variasi data pada kolom kategorikal.

In [ ]:
# Mengecek duplikasi baris
print(f"Jumlah baris duplikat pada df_produk: {df_produk.duplicated().sum()}")
print(f"Jumlah baris duplikat pada df_ingredients: {df_ingredients.duplicated().sum()}")


Jumlah baris duplikat pada df_produk: 0
Jumlah baris duplikat pada df_ingredients: 0


### 📌 Insight Duplikasi Data
Tidak ditemukan anomali duplikasi pada kedua dataset (0 Duplikat). Data sangat unik. Namun, fungsi penghapusan duplikat tetap akan dituliskan di tahap *Cleaning* sebagai *best practice* perlindungan *pipeline* data di masa depan.

### 2.4 Deskripsi Statistik Data Produk (dataset_skin.csv)
Melihat ringkasan statistik deskriptif pada tabel produk. Karena seluruh kolom pada data ini berupa teks (kategorikal), fungsi `describe(include="all")` akan menampilkan:
- **unique**: Jumlah nilai unik/berbeda pada kolom tersebut.
- **top**: Nilai yang paling sering muncul (modus).
- **freq**: Frekuensi kemunculan dari nilai *top* tersebut.

Ini membantu kita melihat sekilas, misalnya, *Brand* apa yang paling banyak di dataset atau *Masalah Kulit* apa yang paling mendominasi.

In [ ]:
print("--- Deskripsi Statistik Data Produk ---")
display(df_produk.describe(include="all"))

--- Deskripsi Statistik Data Produk ---


,ID,Brand,Produk,Jenis Produk,Untuk Kulit,Masalah Kulit,Ukuran,Tipe Bahan Aktif,Tahun Rilis
count,185.000000,185,185,185,185,185,185,185,185.000000
unique,NaN,10,20,8,5,8,4,8,NaN
top,NaN,Avoskin,Ceramide Barrier Moisturizer,Gel,Kombinasi,Kusam,50 ml,Salicylic Acid,NaN
freq,NaN,25,13,30,46,32,54,34,NaN
mean,93.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020.454054
std,53.549043,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.781334
min,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018.000000
25%,47.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2019.000000
50%,93.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020.000000
75%,139.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022.000000


### 2.5 Deskripsi Statistik Data Kandungan (ingredients_category.csv)
Melakukan ringkasan statistik yang sama untuk tabel referensi kandungan (*ingredients*).

Analisis pada bagian ini sangat penting untuk melihat pola fungsi bahan (kolom `function1`) yang paling sering digunakan dalam industri *skincare*, serta melihat sekilas proporsi bahan yang memiliki label peringatan/risiko pada kolom `warning1`.

In [ ]:
print("--- Deskripsi Statistik Data Ingredients ---")
display(df_ingredients.describe(include="all"))

--- Deskripsi Statistik Data Ingredients ---


,ingredient_name,function1,function2,warning1,warning2,ingredient_origin,ingredient_charge
count,747,747,198,142,49,747,747
unique,747,43,33,9,6,3,4
top,Zinc Stearate,Skin Conditioning,Antioxidant,Allergen,Irritant,Synthetic,Non-ionik
freq,1,154,35,72,32,270,591


---
## 3. Cleaning Data
Berdasarkan temuan-temuan di tahap *Assessing*, kita akan melakukan eksekusi pembersihan secara berurutan.


### 3.1 Menghapus Kolom Tidak Relevan
Membuang kolom `ID` dan `Tahun Rilis` dari data produk untuk efisiensi komputasi, karena umur produk tidak menentukan apakah produk tersebut cocok untuk kulit berjerawat atau tidak.

In [ ]:
# Menghapus kolom (errors='ignore' agar aman jika dijalankan berulang)
df_produk_clean = df_produk.drop(columns=['ID', 'Tahun Rilis'], errors='ignore')

print("=== HASIL PENGHAPUSAN KOLOM ===")
print("Kolom yang tersisa pada Data Produk:")
for col in df_produk_clean.columns:
    print(f"- {col}")

print("\n💡 PENJELASAN OUTPUT: Kolom 'ID' dan 'Tahun Rilis' sudah tidak muncul lagi di daftar kolom produk.")

=== HASIL PENGHAPUSAN KOLOM ===
Kolom yang tersisa pada Data Produk:
- Brand
- Produk
- Jenis Produk
- Untuk Kulit
- Masalah Kulit
- Ukuran
- Tipe Bahan Aktif

💡 PENJELASAN OUTPUT: Kolom 'ID' dan 'Tahun Rilis' sudah tidak muncul lagi di daftar kolom produk.


### 📌 Insight Penghapusan Kolom
Dataset produk kini lebih ramping dan hanya berisi *features* yang berpotensi memiliki bobot kuat untuk analisis maupun *filtering* oleh sistem rekomendasi (Brand, Masalah Kulit, Bahan Aktif, dll).

### 3.2 Menangani Data Duplikat (Preventif)
Mengeksekusi fungsi `drop_duplicates()` sebagai bentuk keamanan pencegahan (preventif).

In [ ]:
baris_awal = df_produk_clean.shape[0]

df_produk_clean = df_produk_clean.drop_duplicates()
df_ingredients_clean = df_ingredients.drop_duplicates()

print("=== HASIL PENGHAPUSAN DUPLIKAT ===")
print(f"Data Produk siap     : {df_produk_clean.shape[0]} baris.")
print(f"Data Ingredients siap: {df_ingredients_clean.shape[0]} baris.")
print("\n💡 PENJELASAN OUTPUT: Jumlah baris tetap sama seperti tahap awal karena memang tidak ditemukan duplikat. Data aman.")

=== HASIL PENGHAPUSAN DUPLIKAT ===
Data Produk siap     : 185 baris.
Data Ingredients siap: 747 baris.

💡 PENJELASAN OUTPUT: Jumlah baris tetap sama seperti tahap awal karena memang tidak ditemukan duplikat. Data aman.


### 3.3 Imputasi Missing Values
Mengisi data yang bolong (*NaN/Null*) pada data *Ingredients*.
- Jika tidak ada `warning`, kita isi dengan "**Aman**".
- Jika tidak ada `function2` (fungsi tambahan), kita isi dengan "**Tidak Ada**".

In [ ]:
# Imputasi / Pengisian Nilai
df_ingredients_clean['warning1'] = df_ingredients_clean['warning1'].fillna('Aman')
df_ingredients_clean['warning2'] = df_ingredients_clean['warning2'].fillna('Aman')
df_ingredients_clean['function2'] = df_ingredients_clean['function2'].fillna('Tidak Ada')

print("=== HASIL IMPUTASI MISSING VALUES ===")
print(df_ingredients_clean[['function1', 'function2', 'warning1']].isna().sum())
print("\n💡 PENJELASAN OUTPUT: Angka 0 menunjukkan bahwa imputasi berhasil. Tidak ada lagi nilai 'NaN' yang tertinggal di kolom peringatan maupun fungsi.")

=== HASIL IMPUTASI MISSING VALUES ===
function1    0
function2    0
warning1     0
dtype: int64

💡 PENJELASAN OUTPUT: Angka 0 menunjukkan bahwa imputasi berhasil. Tidak ada lagi nilai 'NaN' yang tertinggal di kolom peringatan maupun fungsi.


### 📌 Insight Imputasi Missing Values
Data kandungan kini 100% lengkap. Dengan mengisi nilai *NaN* menjadi teks definitif seperti "Aman", algoritma klasifikasi *rule-based* nanti dapat membaca kondisinya dengan jelas tanpa memicu *error traceback*.

In [ ]:
# Mengubah teks ke format huruf kecil dan membuang spasi kosong di ujung teks
df_produk_clean['bahan_key'] = df_produk_clean['Tipe Bahan Aktif'].astype(str).str.lower().str.strip()
df_ingredients_clean['bahan_key'] = df_ingredients_clean['ingredient_name'].astype(str).str.lower().str.strip()

print("=== HASIL STANDARISASI TEKS ===")
display(df_produk_clean[['Tipe Bahan Aktif', 'bahan_key']].head(3))
print("\n💡 PENJELASAN OUTPUT: Perhatikan kolom 'bahan_key'. Teks 'Bakuchiol' telah berubah menjadi 'bakuchiol' (huruf kecil semua) dan siap dijadikan jembatan penghubung antar dua dataset.")

=== HASIL STANDARISASI TEKS ===


,Tipe Bahan Aktif,bahan_key
0,Bakuchiol,bakuchiol
1,Hyaluronic Acid,hyaluronic acid
2,Salicylic Acid,salicylic acid



💡 PENJELASAN OUTPUT: Perhatikan kolom 'bahan_key'. Teks 'Bakuchiol' telah berubah menjadi 'bakuchiol' (huruf kecil semua) dan siap dijadikan jembatan penghubung antar dua dataset.


### 📌 Insight Standarisasi & Validasi Akhir Cleaning
Pembersihan telah selesai dilakukan secara menyeluruh (Drop kolom, Drop Duplikat, Imputasi, dan Formatting Key). Kedua dataset (*Produk* dan *Ingredients*) saat ini berada dalam kondisi **100% Matang dan Terstandardisasi**, siap untuk diolah ke dalam tahap *Feature Engineering* dan *Merging*.

In [ ]:
# ============================================================
# VALIDASI & KOREKSI METADATA
# ============================================================

# Memprediksi jenis produk berdasarkan nama produk
def prediksi_jenis_dari_nama(nama_produk):
    nama = str(nama_produk).lower()

    if 'sunscreen' in nama or 'spf' in nama:
        return 'Sunscreen'
    elif 'cleanser' in nama or 'facial wash' in nama or 'face wash' in nama or 'wash' in nama:
        return 'Cleanser'
    elif 'toner' in nama:
        return 'Toner'
    elif 'serum' in nama:
        return 'Serum'
    elif 'ampoule' in nama:
        return 'Serum'
    elif 'essence' in nama:
        return 'Essence'
    elif 'spot gel' in nama or 'spot treatment' in nama or 'acne treatment' in nama:
        return 'Spot Treatment'
    elif 'night cream' in nama or ('night' in nama and 'cream' in nama):
        return 'Night Cream'
    elif'aloe vera gel' in nama or 'hydrating gel' in nama:
        return 'Gel Moisturizer'
    elif 'day cream' in nama or ('day' in nama and 'cream' in nama):
        return 'Day Cream'
    elif 'moisturizer' in nama or 'moisturiser' in nama :
        return 'Moisturizer'
    else:
        return 'Tidak terdeteksi'


# Memprediksi bahan aktif berdasarkan nama produk
def prediksi_bahan_dari_nama(nama_produk):
    nama = str(nama_produk).lower()

    if 'salicylic' in nama:
        return 'Salicylic Acid'
    elif 'hyaluronic' in nama:
        return 'Hyaluronic Acid'
    elif 'ceramide' in nama:
        return 'Ceramide'
    elif 'tea tree' in nama:
        return 'Tea Tree'
    elif 'vitamin c' in nama:
        return 'Vitamin C'
    elif 'retinol' in nama:
        return 'Retinol'
    elif 'bakuchiol' in nama:
        return 'Bakuchiol'
    elif 'niacinamide' in nama:
        return 'Niacinamide'
    elif 'b5' in nama or 'panthenol' in nama:
        return 'Panthenol'
    else:
        return 'Tidak terdeteksi'


# Membuat hasil prediksi dari nama produk
df_produk_clean['Jenis_Produk_Prediksi'] = df_produk_clean['Produk'].apply(prediksi_jenis_dari_nama)
df_produk_clean['Bahan_Aktif_Prediksi'] = df_produk_clean['Produk'].apply(prediksi_bahan_dari_nama)


# Membuat kolom final untuk jenis produk
df_produk_clean['Jenis_Produk_Final'] = df_produk_clean.apply(
    lambda row: row['Jenis_Produk_Prediksi']
    if row['Jenis_Produk_Prediksi'] != 'Tidak terdeteksi'
    else row['Jenis Produk'],
    axis=1
)


# Membuat kolom final untuk bahan aktif
df_produk_clean['Tipe_Bahan_Aktif_Final'] = df_produk_clean.apply(
    lambda row: row['Bahan_Aktif_Prediksi']
    if row['Bahan_Aktif_Prediksi'] != 'Tidak terdeteksi'
    else row['Tipe Bahan Aktif'],
    axis=1
)


# Membuat ulang bahan_key berdasarkan bahan aktif final
df_produk_clean['bahan_key'] = (
    df_produk_clean['Tipe_Bahan_Aktif_Final']
    .astype(str)
    .str.lower()
    .str.strip()
)

# Menampilkan data yang kemungkinan jenis produknya berubah
df_validasi_metadata = df_produk_clean[
    (df_produk_clean['Jenis Produk'] != df_produk_clean['Jenis_Produk_Final']) |
    (df_produk_clean['Tipe Bahan Aktif'] != df_produk_clean['Tipe_Bahan_Aktif_Final'])
]

print("Jumlah data yang mengalami koreksi metadata:", df_validasi_metadata.shape[0])

display(
    df_validasi_metadata[
        [
            'Brand',
            'Produk',
            'Jenis Produk',
            'Jenis_Produk_Final',
            'Tipe Bahan Aktif',
            'Tipe_Bahan_Aktif_Final',
            'bahan_key'
        ]
    ].head(30)
)

Jumlah data yang mengalami koreksi metadata: 168


,Brand,Produk,Jenis Produk,Jenis_Produk_Final,Tipe Bahan Aktif,Tipe_Bahan_Aktif_Final,bahan_key
0,Avoskin,Tea Tree Spot Gel,Essence,Spot Treatment,Bakuchiol,Tea Tree,tea tree
1,Dear Me Beauty,Bakuchiol Night Cream,Essence,Night Cream,Hyaluronic Acid,Bakuchiol,bakuchiol
2,Somethinc,Glow Boost Serum,Gel,Serum,Salicylic Acid,Salicylic Acid,salicylic acid
3,COSRX,Acne Treatment Serum,Cleanser,Serum,Bakuchiol,Bakuchiol,bakuchiol
4,The Ordinary,Tea Tree Spot Gel,Sunscreen,Spot Treatment,Hyaluronic Acid,Tea Tree,tea tree
6,Scarlett,Salicylic Acid Daily Gentle Cleanser,Serum,Cleanser,Hyaluronic Acid,Salicylic Acid,salicylic acid
7,Avoskin,Acne Treatment Serum,Sunscreen,Serum,Ceramide,Ceramide,ceramide
8,Emina,Advanced Snail 96 Mucin Essence,Cream,Essence,Retinol,Retinol,retinol
9,The Ordinary,Bakuchiol Night Cream,Serum,Night Cream,Salicylic Acid,Bakuchiol,bakuchiol
10,Avoskin,Glow Boost Serum,Moisturizer,Serum,Salicylic Acid,Salicylic Acid,salicylic acid


### 📌 Insight validasi dan koreksi metadata

Dilakukan validasi terhadap kesesuaian nama produk, jenis dan tipe bahan aktif.
dari hasil yang ditampilkan, ditemukan banyak produk yang memiliki metadata yang kurang sesuai dengan produk yang sebenarnya.

## 4. Feature Engineering & Merging Data
Data produk dan *ingredients* digabungkan menggunakan metode **Left Join**.
Setelah itu, kita membuat fitur (*kolom*) baru bernama `Rekomendasi_Jerawat` menggunakan pendekatan *Rule-Based System* yang mendeteksi kecocokan bahan aktif produk dengan tingkat keparahan jerawat pengguna (Level 1 hingga Level 4).


**🌟 REVISI LOGIKA UNTUK AI & BACK-END:** Pada tahap ini, kita membuat kolom baru `AI_Target_Level` berisi **angka murni (1, 2, 3, 4)**.
- Bahan aktif yang sangat umum (seperti *Niacinamide*, *Salicylic Acid*, *AHA/BHA*) dipetakan ke target **Level 1 & 2** (Jerawat Ringan - Sedang) agar selaras dengan output prediksi model CNN milik Tim AI dan mencegah *bug* di sistem Back-End.

In [ ]:
# Mengecek jumlah bahan aktif unik pada data produk
jumlah_bahan_produk = df_produk_clean['bahan_key'].nunique()
jumlah_bahan_ingredients = df_ingredients_clean['bahan_key'].nunique()

print("Jumlah bahan aktif unik di data produk:", jumlah_bahan_produk)
print("Jumlah bahan aktif unik di data ingredients:", jumlah_bahan_ingredients)

# Membuat daftar bahan aktif yang ada di produk
bahan_produk = set(df_produk_clean['bahan_key'].unique())

# Membuat daftar bahan aktif yang ada di data ingredients
bahan_ingredients = set(df_ingredients_clean['bahan_key'].unique())

# Mengecek bahan aktif produk yang tidak ditemukan di data ingredients
bahan_tidak_ditemukan = sorted(bahan_produk - bahan_ingredients)

print("\nJumlah bahan aktif produk yang tidak ditemukan di data ingredients:", len(bahan_tidak_ditemukan))

# Menampilkan daftar bahan yang tidak ditemukan
if len(bahan_tidak_ditemukan) > 0:
    display(pd.DataFrame(bahan_tidak_ditemukan, columns=['bahan_key_tidak_ditemukan']))
else:
    print("Semua bahan aktif produk ditemukan di data ingredients.")

Jumlah bahan aktif unik di data produk: 8
Jumlah bahan aktif unik di data ingredients: 747

Jumlah bahan aktif produk yang tidak ditemukan di data ingredients: 4


,bahan_key_tidak_ditemukan
0,bakuchiol
1,ceramide
2,tea tree
3,vitamin c


### 📌 Insight Validasi Bahan Aktif sebelum Merging

Dilakukan pengecekan kesesuaian bahan aktif antara dataset produk dan dataset ingredients. Hasil menunjukkan ada beberapa bahan yang tidak ditemukan di dalam produl, namun setelah ditambahkan bahan aktif secara manual semua data sudah ditemukan.


In [ ]:
# Pengecekan ketersediaan bahan aktif

# Daftar bahan aktif dari produk
bahan_produk = set(df_produk_clean['bahan_key'].unique())

# Daftar bahan aktif dari data ingredients
bahan_ingredients = set(df_ingredients_clean['bahan_key'].unique())

# Cek bahan aktif yang belum ditemukan
bahan_tidak_ditemukan = sorted(bahan_produk - bahan_ingredients)

print("Jumlah bahan aktif final yang belum ditemukan di data ingredients:", len(bahan_tidak_ditemukan))

if len(bahan_tidak_ditemukan) > 0:
    df_bahan_tidak_ditemukan = pd.DataFrame(
        bahan_tidak_ditemukan,
        columns=['bahan_key_tidak_ditemukan']
    )
    display(df_bahan_tidak_ditemukan)
else:
    print("Semua bahan aktif final sudah tersedia di dataset ingredients.")

Jumlah bahan aktif final yang belum ditemukan di data ingredients: 4


,bahan_key_tidak_ditemukan
0,bakuchiol
1,ceramide
2,tea tree
3,vitamin c


### 📌 Insight Validasi Ketersediaan Bahan Aktif
pengecekan bahan akitf final yang mungkin belum tersedia di dataset ingredients, sebagai referensi apabila terjadi penambahana fungsi bahan, warning, dan karakteristik ingredients

In [ ]:
# Menggabungkan data produk dengan data ingredients
# Marging data

df_skincare_merged = df_produk_clean.merge(
    df_ingredients_clean[
        [
            'bahan_key',
            'ingredient_name',
            'function1',
            'function2',
            'warning1',
            'warning2',
            'ingredient_origin',
            'ingredient_charge'
        ]
    ],
    on='bahan_key',
    how='left',
    indicator=True
)

print("=== HASIL MERGE DATA ===")
print("Jumlah baris sebelum merge:", df_produk_clean.shape[0])
print("Jumlah baris setelah merge :", df_skincare_merged.shape[0])

print("\nStatus hasil merge:")
print(df_skincare_merged['_merge'].value_counts())

display(df_skincare_merged.head())

=== HASIL MERGE DATA ===
Jumlah baris sebelum merge: 185
Jumlah baris setelah merge : 185

Status hasil merge:
_merge
both          103
left_only      82
right_only      0
Name: count, dtype: int64


,Brand,Produk,Jenis Produk,Untuk Kulit,Masalah Kulit,Ukuran,Tipe Bahan Aktif,bahan_key,Jenis_Produk_Prediksi,Bahan_Aktif_Prediksi,Jenis_Produk_Final,Tipe_Bahan_Aktif_Final,ingredient_name,function1,function2,warning1,warning2,ingredient_origin,ingredient_charge,_merge
0,Avoskin,Tea Tree Spot Gel,Essence,Berminyak,Jerawat,100 ml,Bakuchiol,tea tree,Spot Treatment,Tea Tree,Spot Treatment,Tea Tree,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
1,Dear Me Beauty,Bakuchiol Night Cream,Essence,Kering,Dehidrasi,50 ml,Hyaluronic Acid,bakuchiol,Night Cream,Bakuchiol,Night Cream,Bakuchiol,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2,Somethinc,Glow Boost Serum,Gel,Berminyak,Flek hitam,30 ml,Salicylic Acid,salicylic acid,Serum,Tidak terdeteksi,Serum,Salicylic Acid,Salicylic Acid,Exfoliant,Tidak Ada,Exfoliating,Irritant,Natural Derivative,Non-ionik,both
3,COSRX,Acne Treatment Serum,Cleanser,Berminyak,Iritasi,100 ml,Bakuchiol,bakuchiol,Serum,Tidak terdeteksi,Serum,Bakuchiol,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
4,The Ordinary,Tea Tree Spot Gel,Sunscreen,Berminyak,Kusam,100 ml,Hyaluronic Acid,tea tree,Spot Treatment,Tea Tree,Spot Treatment,Tea Tree,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


### 📌 Insight Merging data produk dan ingredients
dataset produk dan dataset ingredients digabungkna menggunakan kolom 'bahan_key' sebagai penghubung. proses merge menggunakan left join sehingga seluruh data produk dapat dipertahankan, meskipun ada kemungkinan bahan masih tidak ditemukan pada dataset ingredients.

In [ ]:
# Mengisi data kosong setelah proses merge
kolom_ingredients = [
    'ingredient_name',
    'function1',
    'function2',
    'warning1',
    'warning2',
    'ingredient_origin',
    'ingredient_charge'
]

df_skincare_merged[kolom_ingredients] = df_skincare_merged[kolom_ingredients].fillna(
    'Tidak ditemukan di data referensi'
)

# Menghapus kolom _merge karena hanya dipakai untuk pengecekan
df_skincare_merged = df_skincare_merged.drop(columns=['_merge'])

print("=== CEK MISSING VALUE SETELAH MERGE ===")
print(df_skincare_merged.isna().sum())

display(df_skincare_merged.head())

=== CEK MISSING VALUE SETELAH MERGE ===
Brand                     0
Produk                    0
Jenis Produk              0
Untuk Kulit               0
Masalah Kulit             0
Ukuran                    0
Tipe Bahan Aktif          0
bahan_key                 0
Jenis_Produk_Prediksi     0
Bahan_Aktif_Prediksi      0
Jenis_Produk_Final        0
Tipe_Bahan_Aktif_Final    0
ingredient_name           0
function1                 0
function2                 0
warning1                  0
warning2                  0
ingredient_origin         0
ingredient_charge         0
dtype: int64


,Brand,Produk,Jenis Produk,Untuk Kulit,Masalah Kulit,Ukuran,Tipe Bahan Aktif,bahan_key,Jenis_Produk_Prediksi,Bahan_Aktif_Prediksi,Jenis_Produk_Final,Tipe_Bahan_Aktif_Final,ingredient_name,function1,function2,warning1,warning2,ingredient_origin,ingredient_charge
0,Avoskin,Tea Tree Spot Gel,Essence,Berminyak,Jerawat,100 ml,Bakuchiol,tea tree,Spot Treatment,Tea Tree,Spot Treatment,Tea Tree,Tidak ditemukan di data referensi,Tidak ditemukan di data referensi,Tidak ditemukan di data referensi,Tidak ditemukan di data referensi,Tidak ditemukan di data referensi,Tidak ditemukan di data referensi,Tidak ditemukan di data referensi
1,Dear Me Beauty,Bakuchiol Night Cream,Essence,Kering,Dehidrasi,50 ml,Hyaluronic Acid,bakuchiol,Night Cream,Bakuchiol,Night Cream,Bakuchiol,Tidak ditemukan di data referensi,Tidak ditemukan di data referensi,Tidak ditemukan di data referensi,Tidak ditemukan di data referensi,Tidak ditemukan di data referensi,Tidak ditemukan di data referensi,Tidak ditemukan di data referensi
2,Somethinc,Glow Boost Serum,Gel,Berminyak,Flek hitam,30 ml,Salicylic Acid,salicylic acid,Serum,Tidak terdeteksi,Serum,Salicylic Acid,Salicylic Acid,Exfoliant,Tidak Ada,Exfoliating,Irritant,Natural Derivative,Non-ionik
3,COSRX,Acne Treatment Serum,Cleanser,Berminyak,Iritasi,100 ml,Bakuchiol,bakuchiol,Serum,Tidak terdeteksi,Serum,Bakuchiol,Tidak ditemukan di data referensi,Tidak ditemukan di data referensi,Tidak ditemukan di data referensi,Tidak ditemukan di data referensi,Tidak ditemukan di data referensi,Tidak ditemukan di data referensi,Tidak ditemukan di data referensi
4,The Ordinary,Tea Tree Spot Gel,Sunscreen,Berminyak,Kusam,100 ml,Hyaluronic Acid,tea tree,Spot Treatment,Tea Tree,Spot Treatment,Tea Tree,Tidak ditemukan di data referensi,Tidak ditemukan di data referensi,Tidak ditemukan di data referensi,Tidak ditemukan di data referensi,Tidak ditemukan di data referensi,Tidak ditemukan di data referensi,Tidak ditemukan di data referensi


### 📌 Insight Validasi setelah hasil merding

Dilakukan pengecekan ulang terhadap missing value pada dataset setelah proses penggabungan.
Hasil pengecekan menunjukkan bahwa kedua data sudah tergabung dengan baik.

In [ ]:
# Membuat kolom gabungan teks untuk membantu rule-based system membaca informasi produk
def gabungkan_informasi_produk(row):
    teks = (
        str(row.get('Tipe_Bahan_Aktif_Final', '')) + ' ' +
        str(row.get('Masalah Kulit', '')) + ' ' +
        str(row.get('Untuk Kulit', '')) + ' ' +
        str(row.get('function1', '')) + ' ' +
        str(row.get('function2', '')) + ' ' +
        str(row.get('warning1', '')) + ' ' +
        str(row.get('warning2', ''))
    )
    return teks.lower()

df_skincare_merged['fitur_teks'] = df_skincare_merged.apply(
    gabungkan_informasi_produk,
    axis=1
)

display(
    df_skincare_merged[
        [
            'Brand',
            'Produk',
            'Jenis_Produk_Final',
            'Tipe_Bahan_Aktif_Final',
            'Masalah Kulit',
            'Untuk Kulit',
            'fitur_teks'
        ]
    ].head(10)
)

,Brand,Produk,Jenis_Produk_Final,Tipe_Bahan_Aktif_Final,Masalah Kulit,Untuk Kulit,fitur_teks
0,Avoskin,Tea Tree Spot Gel,Spot Treatment,Tea Tree,Jerawat,Berminyak,tea tree jerawat berminyak tidak ditemukan di data referensi tidak ditemukan di data referensi tidak ditemukan di data referensi tidak ditemukan di data referensi
1,Dear Me Beauty,Bakuchiol Night Cream,Night Cream,Bakuchiol,Dehidrasi,Kering,bakuchiol dehidrasi kering tidak ditemukan di data referensi tidak ditemukan di data referensi tidak ditemukan di data referensi tidak ditemukan di data referensi
2,Somethinc,Glow Boost Serum,Serum,Salicylic Acid,Flek hitam,Berminyak,salicylic acid flek hitam berminyak exfoliant tidak ada exfoliating irritant
3,COSRX,Acne Treatment Serum,Serum,Bakuchiol,Iritasi,Berminyak,bakuchiol iritasi berminyak tidak ditemukan di data referensi tidak ditemukan di data referensi tidak ditemukan di data referensi tidak ditemukan di data referensi
4,The Ordinary,Tea Tree Spot Gel,Spot Treatment,Tea Tree,Kusam,Berminyak,tea tree kusam berminyak tidak ditemukan di data referensi tidak ditemukan di data referensi tidak ditemukan di data referensi tidak ditemukan di data referensi
5,Dear Me Beauty,Acne Treatment Serum,Serum,Hyaluronic Acid,Dehidrasi,Kering,hyaluronic acid dehidrasi kering humectant tidak ada aman aman
6,Scarlett,Salicylic Acid Daily Gentle Cleanser,Cleanser,Salicylic Acid,Flek hitam,Sensitif,salicylic acid flek hitam sensitif exfoliant tidak ada exfoliating irritant
7,Avoskin,Acne Treatment Serum,Serum,Ceramide,Kusam,Kering,ceramide kusam kering tidak ditemukan di data referensi tidak ditemukan di data referensi tidak ditemukan di data referensi tidak ditemukan di data referensi
8,Emina,Advanced Snail 96 Mucin Essence,Essence,Retinol,Iritasi,Kering,retinol iritasi kering skin conditioning tidak ada irritant fotosensitizing
9,The Ordinary,Bakuchiol Night Cream,Night Cream,Bakuchiol,Flek hitam,Normal,bakuchiol flek hitam normal tidak ditemukan di data referensi tidak ditemukan di data referensi tidak ditemukan di data referensi tidak ditemukan di data referensi


### 📌 Insight Pembuatan Fitur Teks
Kolom baru bernama "fitur_teks" ditambahkan dengan berisi gabungan informasi dari beberapa kolom aktif, seperti tipe bahan aktif, masalah kulit, tipe kulit, fungsi bahan, dan warning dari bahan.

In [ ]:
# Fungsi untuk mengecek apakah teks mengandung salah satu keyword
def contains_keyword(text, keywords):
    text = str(text).lower()
    return any(keyword in text for keyword in keywords)


# Keyword untuk kategori rekomendasi
acne_keywords = [
    'acne',
    'jerawat',
    'salicylic',
    'bha',
    'tea tree',
    'niacinamide',
    'benzoyl',
    'retinol',
    'oil control',
    'sebum',
    'pore'
]

soothing_keywords = [
    'soothing',
    'calming',
    'centella',
    'cica',
    'aloe',
    'panthenol',
    'allantoin'
]

hydrating_barrier_keywords = [
    'hydrating',
    'moisturizing',
    'hyaluronic',
    'glycerin',
    'ceramide',
    'barrier'
]

warning_keywords = [
    'irritant',
    'irritation',
    'iritasi',
    'comedogenic',
    'komedogenik',
    'drying',
    'allergen',
    'alergi',
    'patch test'
]


# Membuat kolom fitur boolean
df_skincare_merged['is_acne_care'] = df_skincare_merged['fitur_teks'].apply(
    lambda x: contains_keyword(x, acne_keywords)
)

df_skincare_merged['is_soothing'] = df_skincare_merged['fitur_teks'].apply(
    lambda x: contains_keyword(x, soothing_keywords)
)

df_skincare_merged['is_hydrating_barrier'] = df_skincare_merged['fitur_teks'].apply(
    lambda x: contains_keyword(x, hydrating_barrier_keywords)
)

df_skincare_merged['has_warning'] = df_skincare_merged['fitur_teks'].apply(
    lambda x: contains_keyword(x, warning_keywords)
)

display(
    df_skincare_merged[
        [
            'Produk',
            'Tipe Bahan Aktif',
            'is_acne_care',
            'is_soothing',
            'is_hydrating_barrier',
            'has_warning'
        ]
    ].head(20)
)

,Produk,Tipe Bahan Aktif,is_acne_care,is_soothing,is_hydrating_barrier,has_warning
0,Tea Tree Spot Gel,Bakuchiol,True,False,False,False
1,Bakuchiol Night Cream,Hyaluronic Acid,False,False,False,False
2,Glow Boost Serum,Salicylic Acid,True,False,False,True
3,Acne Treatment Serum,Bakuchiol,False,False,False,True
4,Tea Tree Spot Gel,Hyaluronic Acid,True,False,False,False
5,Acne Treatment Serum,Hyaluronic Acid,False,False,True,False
6,Salicylic Acid Daily Gentle Cleanser,Hyaluronic Acid,True,False,False,True
7,Acne Treatment Serum,Ceramide,False,False,True,False
8,Advanced Snail 96 Mucin Essence,Retinol,True,False,False,True
9,Bakuchiol Night Cream,Salicylic Acid,False,False,False,False


### 📌 Insight Pembuatan fitur boolean
Dilakukan feature dengan membuat fitur boolean, fitur yang dibuat berdasarkan kata kunci yang muncul pada kolom 'fitur_teks'


In [ ]:
# menentukan level jerawat


def tentukan_level_utama(row):
    teks = str(row.get('fitur_teks', '')).lower()

    # Level 1: jerawat ringan
    # Bahan aktif acne care ringan-sedang masuk ke level 1
    if any(k in teks for k in [
        'niacinamide',
        'salicylic',
        'salicylic acid',
        'aha',
        'bha',
        'tea tree',
        'zinc',
        'acne',
        'jerawat',
        'oil control',
        'sebum',
        'pore'
    ]):
        return 1

    # Level 0: kulit normal / sangat ringan / basic care
    elif any(k in teks for k in [
        'hydrating',
        'moisturizing',
        'hyaluronic',
        'glycerin',
        'ceramide',
        'barrier',
        'soothing',
        'calming',
        'centella',
        'cica',
        'aloe',
        'panthenol',
        'allantoin'
    ]):
        return 0

    # Level 2: sedang / perlu perhatian penggunaan
    elif row.get('has_warning', False):
        return 2

    # Level 3: berat / umum sebagai pendukung dan perlu konsultasi
    else:
        return 3

def tentukan_label_level(level):
    mapping = {
        0: 'Tingkat 0 — Kulit normal atau sangat ringan',
        1: 'Tingkat 1 — Jerawat ringan',
        2: 'Tingkat 2 — Jerawat sedang',
        3: 'Tingkat 3 — Jerawat berat'
    }
    return mapping.get(level, 'Tingkat tidak diketahui')


df_skincare_merged['Level_Utama'] = df_skincare_merged.apply(
    tentukan_level_utama,
    axis=1
)

df_skincare_merged['Label_Level'] = df_skincare_merged['Level_Utama'].apply(
    tentukan_label_level
)

display(
    df_skincare_merged[
        [
            'Brand',
            'Produk',
            'Jenis_Produk_Final',
            'Tipe_Bahan_Aktif_Final',
            'Level_Utama',
            'Label_Level'
        ]
    ].head(20)
)


,Brand,Produk,Jenis_Produk_Final,Tipe_Bahan_Aktif_Final,Level_Utama,Label_Level
0,Avoskin,Tea Tree Spot Gel,Spot Treatment,Tea Tree,1,Tingkat 1 — Jerawat ringan
1,Dear Me Beauty,Bakuchiol Night Cream,Night Cream,Bakuchiol,3,Tingkat 3 — Jerawat berat
2,Somethinc,Glow Boost Serum,Serum,Salicylic Acid,1,Tingkat 1 — Jerawat ringan
3,COSRX,Acne Treatment Serum,Serum,Bakuchiol,2,Tingkat 2 — Jerawat sedang
4,The Ordinary,Tea Tree Spot Gel,Spot Treatment,Tea Tree,1,Tingkat 1 — Jerawat ringan
5,Dear Me Beauty,Acne Treatment Serum,Serum,Hyaluronic Acid,0,Tingkat 0 — Kulit normal atau sangat ringan
6,Scarlett,Salicylic Acid Daily Gentle Cleanser,Cleanser,Salicylic Acid,1,Tingkat 1 — Jerawat ringan
7,Avoskin,Acne Treatment Serum,Serum,Ceramide,0,Tingkat 0 — Kulit normal atau sangat ringan
8,Emina,Advanced Snail 96 Mucin Essence,Essence,Retinol,2,Tingkat 2 — Jerawat sedang
9,The Ordinary,Bakuchiol Night Cream,Night Cream,Bakuchiol,3,Tingkat 3 — Jerawat berat


### 📌 Insight Kolom Rekomendasi Jerawat
Kolom dibuat untuk mengubungkan karakteristik produk dengan tingkat keparahan jerawat. Dibuat menggunakan pendekatan rule - based.

Kolom Level Kecocokan digunakan karena satu produk skincare dapat mendukung lebih dari satu kondisi jerawat. Misalnya, produk hydrating/barrier dapat menjadi basic skincare untuk jerawat ringan, tetapi juga dapat berperan sebagai pendukung pada jerawat sangat berat. Dengan adanya kolom level utama dan level kecocokan , sistem tetap memiliki level utama yang ringkas, tetapi fungsi rekomendasi tetap dapat menampilkan produk yang sesuai untuk setiap level yang dipilih.

In [ ]:
def hitung_skor_rekomendasi(row):
    skor = 0

    if row.get('is_acne_care', False):
        skor += 3

    if row.get('is_soothing', False):
        skor += 2

    if row.get('is_hydrating_barrier', False):
        skor += 2

    if row.get('has_warning', False):
        skor -= 1

    return skor


df_skincare_merged['Skor_Rekomendasi'] = df_skincare_merged.apply(
    hitung_skor_rekomendasi,
    axis=1
)

df_skincare_merged[
    [
        'Brand',
        'Produk',
        'Jenis_Produk_Final',
        'Tipe_Bahan_Aktif_Final',
        'Level_Utama',
        'Label_Level',
        'Skor_Rekomendasi'
    ]
].head(20)

,Brand,Produk,Jenis_Produk_Final,Tipe_Bahan_Aktif_Final,Level_Utama,Label_Level,Skor_Rekomendasi
0,Avoskin,Tea Tree Spot Gel,Spot Treatment,Tea Tree,1,Tingkat 1 — Jerawat ringan,3
1,Dear Me Beauty,Bakuchiol Night Cream,Night Cream,Bakuchiol,3,Tingkat 3 — Jerawat berat,0
2,Somethinc,Glow Boost Serum,Serum,Salicylic Acid,1,Tingkat 1 — Jerawat ringan,2
3,COSRX,Acne Treatment Serum,Serum,Bakuchiol,2,Tingkat 2 — Jerawat sedang,-1
4,The Ordinary,Tea Tree Spot Gel,Spot Treatment,Tea Tree,1,Tingkat 1 — Jerawat ringan,3
5,Dear Me Beauty,Acne Treatment Serum,Serum,Hyaluronic Acid,0,Tingkat 0 — Kulit normal atau sangat ringan,2
6,Scarlett,Salicylic Acid Daily Gentle Cleanser,Cleanser,Salicylic Acid,1,Tingkat 1 — Jerawat ringan,2
7,Avoskin,Acne Treatment Serum,Serum,Ceramide,0,Tingkat 0 — Kulit normal atau sangat ringan,2
8,Emina,Advanced Snail 96 Mucin Essence,Essence,Retinol,2,Tingkat 2 — Jerawat sedang,2
9,The Ordinary,Bakuchiol Night Cream,Night Cream,Bakuchiol,3,Tingkat 3 — Jerawat berat,0


### 📌 Insight Membuat skor rekomendasi
Untuk menentukan prioritas produk, di mana produk akan mendapatkan skor tambahan jika memiliki karakteristik yang relevan.
Produk dengan skor rendah tidak dijadikan rekomendasi utama karena meskipun target levelnya sesuai, produk tersebut tidak memiliki cukup indikator acne care, soothing, atau hydrating barrier, atau memiliki warning penggunaan.

In [ ]:
def buat_catatan_berdasarkan_level(level):
    # Mengubah data menjadi string (misal: angka 1 menjadi '1')
    level_teks = str(level).lower()

    # Perbaikan: Langsung cek karakter angkanya saja ('1', '2', '3', '4')
    if '1' in level_teks:
        return 'Produk cocok untuk perawatan jerawat ringan karena mendukung basic skincare, soothing, atau hidrasi kulit.'

    elif '2' in level_teks:
        return 'Produk dapat dipertimbangkan untuk jerawat sedang karena memiliki kandungan atau klaim yang berkaitan dengan acne care.'

    elif '3' in level_teks:
        return 'Produk dapat membantu mendukung perawatan jerawat berat dengan fokus menjaga skin barrier dan menenangkan kulit.'

    elif '4' in level_teks:
        return 'Produk hanya sebagai basic skincare pendukung. Untuk jerawat sangat berat, pengguna tetap disarankan berkonsultasi dengan dokter kulit.'

    else:
        return 'Produk bersifat umum dan tidak secara khusus ditujukan untuk jerawat.'

# Menerapkan fungsi yang sudah diperbaiki
df_skincare_merged['Catatan_Rekomendasi'] = df_skincare_merged['Level_Utama'].apply(
    buat_catatan_berdasarkan_level,
)

# Menampilkan hasil cetak di notebook untuk memastikan catatan sudah bervariasi
display(
    df_skincare_merged[
        [
            'Brand',
            'Produk',
            'Jenis_Produk_Final',
            'Tipe_Bahan_Aktif_Final',
            'Level_Utama',
            'Skor_Rekomendasi',
            'Catatan_Rekomendasi'
        ]
    ].head(20)
)

,Brand,Produk,Jenis_Produk_Final,Tipe_Bahan_Aktif_Final,Level_Utama,Skor_Rekomendasi,Catatan_Rekomendasi
0,Avoskin,Tea Tree Spot Gel,Spot Treatment,Tea Tree,1,3,"Produk cocok untuk perawatan jerawat ringan karena mendukung basic skincare, soothing, atau hidrasi kulit."
1,Dear Me Beauty,Bakuchiol Night Cream,Night Cream,Bakuchiol,3,0,Produk dapat membantu mendukung perawatan jerawat berat dengan fokus menjaga skin barrier dan menenangkan kulit.
2,Somethinc,Glow Boost Serum,Serum,Salicylic Acid,1,2,"Produk cocok untuk perawatan jerawat ringan karena mendukung basic skincare, soothing, atau hidrasi kulit."
3,COSRX,Acne Treatment Serum,Serum,Bakuchiol,2,-1,Produk dapat dipertimbangkan untuk jerawat sedang karena memiliki kandungan atau klaim yang berkaitan dengan acne care.
4,The Ordinary,Tea Tree Spot Gel,Spot Treatment,Tea Tree,1,3,"Produk cocok untuk perawatan jerawat ringan karena mendukung basic skincare, soothing, atau hidrasi kulit."
5,Dear Me Beauty,Acne Treatment Serum,Serum,Hyaluronic Acid,0,2,Produk bersifat umum dan tidak secara khusus ditujukan untuk jerawat.
6,Scarlett,Salicylic Acid Daily Gentle Cleanser,Cleanser,Salicylic Acid,1,2,"Produk cocok untuk perawatan jerawat ringan karena mendukung basic skincare, soothing, atau hidrasi kulit."
7,Avoskin,Acne Treatment Serum,Serum,Ceramide,0,2,Produk bersifat umum dan tidak secara khusus ditujukan untuk jerawat.
8,Emina,Advanced Snail 96 Mucin Essence,Essence,Retinol,2,2,Produk dapat dipertimbangkan untuk jerawat sedang karena memiliki kandungan atau klaim yang berkaitan dengan acne care.
9,The Ordinary,Bakuchiol Night Cream,Night Cream,Bakuchiol,3,0,Produk dapat membantu mendukung perawatan jerawat berat dengan fokus menjaga skin barrier dan menenangkan kulit.


### 📌 Insight Catatan Rekomendasi.
Untuk memberikan penjelasan singkat terhadap hasil rekomendasi.
Catatan rekomendasi juga memberikan batasan bahwa skincare hanya berfungsi sebagai pendukung perawatan kulit. Untuk jerawat yang lebih berat, sistem tetap memberikan arahan agar pengguna berkonsultasi dengan dokter kulit.

In [ ]:
def ambil_rekomendasi_berdasarkan_level(level, jumlah_produk=10):
    # Mengubah input level menjadi angka
    level_angka = int(level)

    # Filter produk berdasarkan level angka
    hasil = df_skincare_merged[
        df_skincare_merged["Level_Utama"] == level_angka
    ].copy()

    # Mengurutkan produk berdasarkan skor rekomendasi
    hasil = hasil.sort_values(
        by="Skor_Rekomendasi",
        ascending=False
    )

    # Membuat kolom level yang sedang dipilih
    hasil["Level_Dipilih"] = level_angka

    # Membuat catatan sesuai level yang sedang dipilih
    hasil["Catatan_Level_Dipilih"] = buat_catatan_berdasarkan_level(level_angka)

    kolom_tampil = [
        "Brand",
        "Produk",
        "Jenis_Produk_Final",
        "Tipe_Bahan_Aktif_Final",
        "Level_Dipilih",
        "Level_Utama",
        "Skor_Rekomendasi",
        "Catatan_Level_Dipilih"
    ]

    return hasil[kolom_tampil].head(jumlah_produk)

### 📌 Insight Fungsi Pengambilan Rekomendasi Berdasarkan Level

Dibuat fungsi untuk mengambil rekomendasi produk berdasarkan level jerawat tertentu. Fungsi ini bekerja dengan cara memfilter produk yang sesuai dengan level jerawat, kemudian mengurutkannya berdasarkan `Skor_Rekomendasi`.

Fungsi ini menjadi penghubung antara hasil klasifikasi jerawat dan dataset skincare. Nantinya, ketika model CNN menghasilkan prediksi tingkat jerawat, sistem dapat langsung mengambil daftar produk skincare yang sesuai dengan level tersebut.

In [ ]:
ambil_rekomendasi_berdasarkan_level(3)

,Brand,Produk,Jenis_Produk_Final,Tipe_Bahan_Aktif_Final,Level_Dipilih,Level_Utama,Skor_Rekomendasi,Catatan_Level_Dipilih
1,Dear Me Beauty,Bakuchiol Night Cream,Night Cream,Bakuchiol,3,3,0,Produk dapat membantu mendukung perawatan jerawat berat dengan fokus menjaga skin barrier dan menenangkan kulit.
9,The Ordinary,Bakuchiol Night Cream,Night Cream,Bakuchiol,3,3,0,Produk dapat membantu mendukung perawatan jerawat berat dengan fokus menjaga skin barrier dan menenangkan kulit.
17,Emina,Acne Treatment Serum,Serum,Bakuchiol,3,3,0,Produk dapat membantu mendukung perawatan jerawat berat dengan fokus menjaga skin barrier dan menenangkan kulit.
25,Avoskin,Vitamin C Serum,Serum,Vitamin C,3,3,0,Produk dapat membantu mendukung perawatan jerawat berat dengan fokus menjaga skin barrier dan menenangkan kulit.
40,Dear Me Beauty,Bakuchiol Night Cream,Night Cream,Bakuchiol,3,3,0,Produk dapat membantu mendukung perawatan jerawat berat dengan fokus menjaga skin barrier dan menenangkan kulit.
43,The Ordinary,Advanced Snail 96 Mucin Essence,Essence,Vitamin C,3,3,0,Produk dapat membantu mendukung perawatan jerawat berat dengan fokus menjaga skin barrier dan menenangkan kulit.
48,COSRX,Vitamin C Serum,Serum,Vitamin C,3,3,0,Produk dapat membantu mendukung perawatan jerawat berat dengan fokus menjaga skin barrier dan menenangkan kulit.
54,Dear Me Beauty,Advanced Snail 96 Mucin Essence,Essence,Vitamin C,3,3,0,Produk dapat membantu mendukung perawatan jerawat berat dengan fokus menjaga skin barrier dan menenangkan kulit.
62,COSRX,Bakuchiol Night Cream,Night Cream,Bakuchiol,3,3,0,Produk dapat membantu mendukung perawatan jerawat berat dengan fokus menjaga skin barrier dan menenangkan kulit.
74,Scarlett,Bakuchiol Night Cream,Night Cream,Bakuchiol,3,3,0,Produk dapat membantu mendukung perawatan jerawat berat dengan fokus menjaga skin barrier dan menenangkan kulit.


In [ ]:
ambil_rekomendasi_berdasarkan_level(2)

,Brand,Produk,Jenis_Produk_Final,Tipe_Bahan_Aktif_Final,Level_Dipilih,Level_Utama,Skor_Rekomendasi,Catatan_Level_Dipilih
8,Emina,Advanced Snail 96 Mucin Essence,Essence,Retinol,2,2,2,Produk dapat dipertimbangkan untuk jerawat sedang karena memiliki kandungan atau klaim yang berkaitan dengan acne care.
11,Somethinc,Miraculous Retinol Ampoule,Serum,Retinol,2,2,2,Produk dapat dipertimbangkan untuk jerawat sedang karena memiliki kandungan atau klaim yang berkaitan dengan acne care.
34,Wardah,Retinol Night Serum,Serum,Retinol,2,2,2,Produk dapat dipertimbangkan untuk jerawat sedang karena memiliki kandungan atau klaim yang berkaitan dengan acne care.
18,Avoskin,Retinol Night Serum,Serum,Retinol,2,2,2,Produk dapat dipertimbangkan untuk jerawat sedang karena memiliki kandungan atau klaim yang berkaitan dengan acne care.
38,COSRX,Retinol Night Serum,Serum,Retinol,2,2,2,Produk dapat dipertimbangkan untuk jerawat sedang karena memiliki kandungan atau klaim yang berkaitan dengan acne care.
36,Dear Me Beauty,Cica Soothing Cream,Essence,Retinol,2,2,2,Produk dapat dipertimbangkan untuk jerawat sedang karena memiliki kandungan atau klaim yang berkaitan dengan acne care.
88,COSRX,Miraculous Retinol Ampoule,Serum,Retinol,2,2,2,Produk dapat dipertimbangkan untuk jerawat sedang karena memiliki kandungan atau klaim yang berkaitan dengan acne care.
96,Scarlett,Green Tea Balancing Toner,Toner,Retinol,2,2,2,Produk dapat dipertimbangkan untuk jerawat sedang karena memiliki kandungan atau klaim yang berkaitan dengan acne care.
79,The Ordinary,Miraculous Retinol Ampoule,Serum,Retinol,2,2,2,Produk dapat dipertimbangkan untuk jerawat sedang karena memiliki kandungan atau klaim yang berkaitan dengan acne care.
81,Somethinc,Retinol Night Serum,Serum,Retinol,2,2,2,Produk dapat dipertimbangkan untuk jerawat sedang karena memiliki kandungan atau klaim yang berkaitan dengan acne care.


In [ ]:
ambil_rekomendasi_berdasarkan_level(1)

,Brand,Produk,Jenis_Produk_Final,Tipe_Bahan_Aktif_Final,Level_Dipilih,Level_Utama,Skor_Rekomendasi,Catatan_Level_Dipilih
20,Skintific,Hydrating Aloe Vera Gel,Gel Moisturizer,Ceramide,1,1,5,"Produk cocok untuk perawatan jerawat ringan karena mendukung basic skincare, soothing, atau hidrasi kulit."
47,Scarlett,Luminous Whitening Night Cream,Night Cream,Ceramide,1,1,5,"Produk cocok untuk perawatan jerawat ringan karena mendukung basic skincare, soothing, atau hidrasi kulit."
171,Skintific,Hydra Boost Essence,Essence,Ceramide,1,1,5,"Produk cocok untuk perawatan jerawat ringan karena mendukung basic skincare, soothing, atau hidrasi kulit."
152,Scarlett,Ceramide Barrier Moisturizer,Moisturizer,Ceramide,1,1,5,"Produk cocok untuk perawatan jerawat ringan karena mendukung basic skincare, soothing, atau hidrasi kulit."
94,Avoskin,Ceramide Barrier Moisturizer,Moisturizer,Ceramide,1,1,5,"Produk cocok untuk perawatan jerawat ringan karena mendukung basic skincare, soothing, atau hidrasi kulit."
116,Scarlett,Hyaluronic B5 Serum,Serum,Hyaluronic Acid,1,1,5,"Produk cocok untuk perawatan jerawat ringan karena mendukung basic skincare, soothing, atau hidrasi kulit."
57,Sensatia Botanicals,Ceramide Barrier Moisturizer,Moisturizer,Ceramide,1,1,5,"Produk cocok untuk perawatan jerawat ringan karena mendukung basic skincare, soothing, atau hidrasi kulit."
58,Emina,Exfoliating Toner,Toner,Hyaluronic Acid,1,1,5,"Produk cocok untuk perawatan jerawat ringan karena mendukung basic skincare, soothing, atau hidrasi kulit."
162,Sensatia Botanicals,Hyaluronic B5 Serum,Serum,Hyaluronic Acid,1,1,5,"Produk cocok untuk perawatan jerawat ringan karena mendukung basic skincare, soothing, atau hidrasi kulit."
169,Dear Me Beauty,Green Tea Balancing Toner,Toner,Hyaluronic Acid,1,1,5,"Produk cocok untuk perawatan jerawat ringan karena mendukung basic skincare, soothing, atau hidrasi kulit."


In [ ]:
ambil_rekomendasi_berdasarkan_level(4)

,Brand,Produk,Jenis_Produk_Final,Tipe_Bahan_Aktif_Final,Level_Dipilih,Level_Utama,Skor_Rekomendasi,Catatan_Level_Dipilih


### 📌 Insight uji coba rekomendasi
Uji coba fungsi rekomendasi menunjukkan bahwa sistem sudah mampu menampilkan produk berdasarkan level jerawat yang dipilih, yaitu Level 1, Level 2, Level 3, dan Level 4.
Dari hasil ini dapat disimpulkan bahwa sistem rekomendasi sudah berjalan dengan baik. Meskipun beberapa produk muncul pada lebih dari satu level, hal tersebut masih wajar karena satu produk skincare dapat memiliki fungsi yang relevan untuk beberapa kondisi jerawat. Untuk tampilan akhir, kolom `Level_Dipilih` dan `Catatan_Level_Dipilih` sudah  membuat hasil rekomendasi lebih jelas bagi pengguna.

## 5. Exploratory Data Analysis (EDA)
Tahap EDA dilakukan untuk memahami karakteristik dataset secara mendalam, menemukan pola tersembunyi (*hidden patterns*), dan menjawab **Pertanyaan Bisnis Capstone**.

Analisis dibagi menjadi dua bagian:
1. **Univariate Analysis**: Memahami distribusi masing-masing fitur secara individu.
2. **Bivariate Analysis**: Memahami korelasi atau hubungan antar dua fitur (misal: Level Jerawat vs Jenis Produk).

### 5.1 Univariate Analysis: Keseimbangan Kelas Target AI (AI_Target_Level)
Langkah pertama yang paling krusial sebelum data diumpankan ke model sistem rekomendasi (ataupun diekspor ke aplikasi) adalah melihat distribusi target kelasnya. Apakah data kita didominasi oleh satu level jerawat saja?

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Define df_skincare_ready to resolve NameError
kolom_final = [
    'Brand',
    'Produk',
    'Jenis_Produk_Final',
    'Tipe_Bahan_Aktif_Final',
    'Masalah Kulit',
    'Untuk Kulit',
    'ingredient_name',
    'function1',
    'function2',
    'warning1',
    'warning2',
    'Level_Utama',
    'Label_Level',
    'Skor_Rekomendasi',
    'Catatan_Rekomendasi'
]

df_skincare_ready = df_skincare_merged[kolom_final].copy()

# EDA 1: Distribusi Target Rekomendasi
plt.figure(figsize=(10, 6))

# Menggunakan df_skincare_ready dan kolom 'Label_Level' untuk distribusi
kategori_order = df_skincare_ready['Label_Level'].value_counts().index.tolist()

ax = sns.countplot(data=df_skincare_ready, y='Label_Level',
                   order=kategori_order,
                   palette='viridis',
                   hue='Label_Level',  # Menambahkan hue
                   legend=False)      # Menambahkan legend=False

plt.title('Distribusi Target Rekomendasi (Level Jerawat)', fontsize=15, fontweight='bold')
plt.xlabel('Jumlah Produk', fontsize=12)
plt.ylabel('Kategori Rekomendasi', fontsize=12)

for p in ax.patches:
    ax.annotate(f'{int(p.get_width())}', (p.get_width() + 2, p.get_y() + 0.5), va='center', fontsize=11)

plt.tight_layout()
plt.show()

### 5.2 Univariate Analysis: Top 10 Bahan Aktif Skincare (Market Trend)
Menjawab **Pertanyaan Bisnis No. 3**: Melihat bahan aktif apa yang mendominasi pasar perawatan kulit berjerawat saat ini.

In [ ]:
# EDA 2: Top 10 Bahan Aktif Terbanyak
plt.figure(figsize=(10, 6))
top_bahan = df_skincare_ready['Tipe_Bahan_Aktif_Final'].value_counts().head(10)

ax = sns.barplot(x=top_bahan.values, y=top_bahan.index,
                   palette='mako',
                   hue=top_bahan.index,  # Menambahkan hue
                   legend=False)         # Menambahkan legend=False

plt.title('Top 10 Kandungan Aktif Skincare Terbanyak di Dataset', fontsize=15, fontweight='bold')
plt.xlabel('Jumlah Produk', fontsize=12)
plt.ylabel('Jenis Bahan Aktif', fontsize=12)

for p in ax.patches:
    ax.annotate(f'{int(p.get_width())}', (p.get_width() + 1, p.get_y() + 0.5), va='center')

plt.tight_layout()
plt.show()

### 5.3 Univariate Analysis: Komposisi 'Jenis Produk'
Menganalisis jenis *skincare* (Toner, Serum, Cleanser, dll) apa yang paling banyak diformulasikan.

In [ ]:
# EDA 3: Distribusi Jenis Produk
plt.figure(figsize=(10, 6))
jenis_counts = df_skincare_ready['Jenis_Produk_Final'].value_counts()

ax = sns.barplot(x=jenis_counts.values, y=jenis_counts.index,
                   palette='flare',
                   hue=jenis_counts.index,  # Menambahkan hue
                   legend=False)            # Menambahkan legend=False

plt.title('Distribusi Jenis Produk Skincare', fontsize=15, fontweight='bold')
plt.xlabel('Jumlah Produk', fontsize=12)
plt.ylabel('Jenis Produk', fontsize=12)

plt.tight_layout()
plt.show()

### 5.4 Univariate Analysis: Proporsi Risiko Keamanan
Menjawab **Pertanyaan Bisnis No. 6**: Seberapa besar rasio produk yang berpotensi memicu efek samping (alergen/iritan)?

In [ ]:
# EDA 4: Proporsi Keamanan Kandungan
plt.figure(figsize=(7, 7))
status_keamanan = df_skincare_merged['has_warning'].value_counts()

plt.pie(status_keamanan.values, labels=['Tidak Ada Peringatan', 'Ada Peringatan'], autopct='%1.1f%%',
        startangle=90, colors=['#66b3ff', '#ff9999'], explode=(0.1, 0), shadow=True)

plt.title('Proporsi Keamanan Kandungan Skincare', fontsize=15, fontweight='bold')
plt.show()

### 5.5 Bivariate Analysis: Hubungan Jenis Produk & Level Jerawat
Analisis mendalam untuk melihat apakah produk jerawat Level 4 (Kistik/Parah) lebih banyak didominasi oleh Moisturizer, atau merata.

In [ ]:
# EDA 5: Bivariate Jenis Produk vs Level Rekomendasi
plt.figure(figsize=(12, 7))

# 🌟 FIX ERROR: Mencegah ValueError ambiguity dengan .tolist()
jenis_order = df_skincare_ready['Jenis_Produk_Final'].value_counts().index.tolist()

# Define the order for 'Label_Level' from smallest to largest/most severe
hue_order = [
    'Tingkat 0 — Kulit normal atau sangat ringan',
    'Tingkat 1 — Jerawat ringan',
    'Tingkat 2 — Jerawat sedang',
    'Tingkat 3 — Jerawat berat'
]

ax = sns.countplot(data=df_skincare_ready, y='Jenis_Produk_Final', hue='Label_Level',
                   order=jenis_order, hue_order=hue_order, palette='Set2')

plt.title('Sebaran Kategori Level Jerawat pada Setiap Jenis Produk', fontsize=15, fontweight='bold')
plt.xlabel('Jumlah Produk', fontsize=12)
plt.ylabel('Jenis Produk', fontsize=12)
plt.legend(title='Level Jerawat', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

### 5.6 Bivariate Analysis: Profil Keamanan pada Tiap Level Jerawat
Melihat apakah produk jerawat yang lebih parah memiliki risiko bahan kimia yang lebih tinggi.

In [ ]:
# EDA 6: Bivariate Risiko Keamanan vs Level Rekomendasi
plt.figure(figsize=(14, 8)) # Increased width and height for better readability

# Define the order for 'Label_Level' from smallest to largest/most severe
hue_order_levels = [
    'Tingkat 0 — Kulit normal atau sangat ringan',
    'Tingkat 1 — Jerawat ringan',
    'Tingkat 2 — Jerawat sedang',
    'Tingkat 3 — Jerawat berat'
]

ax = sns.countplot(data=df_skincare_merged,
              y='Label_Level',
              hue='has_warning',
              order=hue_order_levels,
              palette={False: '#66b3ff', True: '#ff9999'})

plt.title('Profil Keamanan Produk Berdasarkan Level Jerawat', fontsize=15, fontweight='bold')
plt.xlabel('Jumlah Produk', fontsize=12)
plt.ylabel('Kategori Rekomendasi', fontsize=12)
plt.legend(title='Risiko Iritasi', bbox_to_anchor=(1.05, 1), loc='upper left', labels=['Aman', 'Ya (Risiko Iritasi)']) # Moved legend outside

# Add annotations to the bars
for p in ax.patches:
    ax.annotate(f'{int(p.get_width())}', # get_width() for horizontal bars
                (p.get_width(), p.get_y() + p.get_height() / 2.),
                ha='left', va='center', fontsize=9, color='black', xytext=(5, 0), # xytxt (x,y) offset
                textcoords='offset points')

plt.tight_layout()
plt.show()

## 6. Export Data
Tahap terakhir adalah menyimpan *dataframe* final yang telah bersih dan berlabel ke dalam bentuk `.csv`. File ini dinamakan `dataset_skincare_ready.csv` dan menjadi output akhir dari tim Data Science untuk dikonsumsi oleh aplikasi.

In [ ]:
kolom_final = [
    'Brand',
    'Produk',
    'Jenis_Produk_Final',
    'Tipe_Bahan_Aktif_Final',
    'Masalah Kulit',
    'Untuk Kulit',
    'ingredient_name',
    'function1',
    'function2',
    'warning1',
    'warning2',
    'Level_Utama',
    'Label_Level',
    'Skor_Rekomendasi',
    'Catatan_Rekomendasi'
]

df_skincare_ready = df_skincare_merged[kolom_final].copy()

df_skincare_ready.to_csv(
    'dataset_skincare_ready.csv',
    index=False
)

print("Dataset final berhasil disimpan sebagai dataset_skincare_ready.csv")
print("Jumlah data final:", df_skincare_ready.shape)

display(df_skincare_ready.head())

## Data Dictionary dan Ringkasan Dataset Rekomendasi Skincare

Dataset ini merupakan data produk skincare yang telah melalui proses cleaning, penghapusan kolom tidak relevan, imputasi missing value, standarisasi teks, penggabungan dengan data ingredients, penentuan level rekomendasi, dan pembuatan skor rekomendasi. Dataset final digunakan sebagai dasar sistem rekomendasi skincare berdasarkan level jerawat hasil klasifikasi model.

### Ringkasan Dataset Setelah Preprocessing

| Informasi | Hasil |
|---|---:|
| Jumlah data produk awal | 185 |
| Jumlah kolom produk awal | 9 |
| Jumlah data ingredients awal | 747 |
| Jumlah kolom ingredients awal | 7 |
| Jumlah duplikat produk | 0 |
| Jumlah duplikat ingredients | 0 |
| Jumlah data final | 185 |
| Jumlah kolom final | 15 |

### Data Dictionary Dataset Rekomendasi Skincare

| Nama Kolom | Tipe Data | Deskripsi |
|---|---|---|
| `Brand` | String/Object | Nama merek produk skincare. |
| `Produk` | String/Object | Nama produk skincare. |
| `Jenis_Produk_Final` | String/Object | Jenis produk skincare setelah divalidasi atau dikoreksi berdasarkan nama produk, misalnya serum, cleanser, toner, sunscreen, moisturizer, atau spot treatment. |
| `Tipe_Bahan_Aktif_Final` | String/Object | Bahan aktif utama produk setelah distandarisasi dan divalidasi, misalnya salicylic acid, tea tree, retinol, ceramide, niacinamide, hyaluronic acid, bakuchiol, atau vitamin C. |
| `Masalah Kulit` | String/Object | Masalah kulit yang menjadi target produk, seperti jerawat, kusam, dehidrasi, iritasi, flek hitam, atau pori besar. |
| `Untuk Kulit` | String/Object | Jenis kulit yang sesuai dengan produk, seperti berminyak, kering, normal, kombinasi, atau sensitif. |
| `ingredient_name` | String/Object | Nama bahan aktif yang berasal dari dataset referensi ingredients. Kolom ini digunakan untuk mencocokkan bahan aktif produk dengan informasi fungsi dan warning. |
| `function1` | String/Object | Fungsi utama dari bahan aktif skincare. |
| `function2` | String/Object | Fungsi tambahan dari bahan aktif. Jika tidak tersedia, nilainya diisi dengan `Tidak Ada`. |
| `warning1` | String/Object | Peringatan utama dari bahan aktif. Jika tidak ada peringatan, nilainya diisi dengan `Aman`. |
| `warning2` | String/Object | Peringatan tambahan dari bahan aktif. Jika tidak ada peringatan, nilainya diisi dengan `Aman`. |
| `Level_Utama` | Integer | Level jerawat utama yang cocok dengan produk berdasarkan rule-based system. Nilainya terdiri dari 0, 1, 2, dan 3. |
| `Label_Level` | String/Object | Keterangan teks dari `Level_Utama`, misalnya `Tingkat 1 — Jerawat ringan`. |
| `Skor_Rekomendasi` | Integer/Float | Skor prioritas rekomendasi produk. Skor dihitung berdasarkan indikator acne care, soothing, hydrating/barrier, dan warning. |
| `Catatan_Rekomendasi` | String/Object | Penjelasan singkat alasan produk direkomendasikan untuk level jerawat tertentu. |

### Penjelasan Level Rekomendasi

| Level | Keterangan |
|---|---|
| 0 | Kulit normal atau jerawat sangat ringan |
| 1 | Jerawat ringan |
| 2 | Jerawat sedang |
| 3 | Jerawat berat |

### Kesimpulan

Berdasarkan hasil preprocessing, dataset rekomendasi skincare telah dibersihkan dan disesuaikan agar dapat digunakan dalam sistem rekomendasi. Dataset akhir terdiri dari 185 produk skincare dengan 15 kolom final. Setiap produk telah memiliki informasi brand, nama produk, jenis produk, bahan aktif, masalah kulit, jenis kulit, fungsi ingredients, warning, level rekomendasi, skor rekomendasi, dan catatan rekomendasi. Data ini digunakan untuk memberikan rekomendasi skincare yang sesuai dengan level jerawat pengguna.
